# Урок 12 - Скорочення історії чату за допомогою Agent Scratchpad

Цей ноутбук демонструє, як керувати контекстом у довгих розмовах за допомогою Microsoft Agent Framework. Зі збільшенням розмови кількість токенів зростає — зрештою перевищуючи вікно контексту моделі. Ми вирішуємо це за допомогою **шаблону узагальнення контексту** та **Agent Scratchpad** для постійної пам’яті.

## Чому ви навчитесь:
1. **Чому важливе управління контекстом**: розуміння обмежень токенів і вікон контексту
2. **Агенти, що враховують контекст**: створення агентів, які керують власним контекстом розмови
3. **Шаблон узагальнення контексту**: використання інструментів для стискання історії розмови
4. **Agent Scratchpad**: постійна пам’ять, що зберігається під час скорочення контексту

## Необхідні умови:
- Налаштування Azure OpenAI з конфігурованими змінними середовища
- Розуміння базових концепцій агентів з попередніх уроків


## Налаштування


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Чому важливо керувати контекстом

Кожен LLM має обмежене **вікно контексту** — максимальну кількість токенів, які він може обробити за один запит. У міру розвитку багатокрокової розмови:

- **Кількість токенів зростає лінійно** з кожним повідомленням користувача і відповіддю асистента.
- **Токени підказки домінують у вартості**, бо вся історія надсилається щоразу заново.
- З часом розмова **перевищує вікно контексту**, і модель або обрізає, або видає помилку.

### Стратегії управління контекстом

| Стратегія | Як працює | Компроміс |
|---|---|---|
| **Обрізка** | Видалення найстаріших повідомлень | Втрачається ранній контекст |
| **Резюмування** | Стискання старих повідомлень у резюме | Втрачається частина деталей, але зберігаються ключові моменти |
| **Записник / Зовнішня пам’ять** | Зберігання ключових фактів поза розмовою | Потрібні виклики інструментів, але витримує будь-яке скорочення |

У цій записці ми поєднуємо **резюмування** з **інструментом записника**, щоб агент міг підтримувати неперервність, навіть коли історія розмови стискається.


## Створення агента з урахуванням контексту


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Моделювання тривалої розмови

Давайте пройдемося через багатокрокову розмову, щоб побачити, як накопичується контекст. Агент має зберігати ключові деталі (уподобання, бюджет, дати подорожі) протягом усієї розмови та демонструвати послідовність.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Зверніть увагу, як агент зберігає контекст з попередніх обмінів — він знає про Японію, суші, храми, фотографію, бюджет $3000, подорож наодинці та період у квітні. У короткій розмові це працює добре, але з розширенням розмови надсилати всю історію стає дорого.

Продовжимо розмову з більшою кількістю обмінів, щоб побачити накопичення контексту:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Шаблон підсумовування контексту

У міру розвитку розмови ми можемо використовувати **інструмент підсумовування**, щоб стиснути накопичений контекст у компактний формат. Агент викликає цей інструмент для запису ключових уподобань, щоб навіть при видаленні старих повідомлень була збережена основна інформація.

Цей шаблон є будівельним блоком для більш складного скорочення історії:
1. Агент визначає ключові факти з розмови
2. Викликає інструмент підсумовування, щоб зберегти їх
3. Старі повідомлення можна безпечно видалити, оскільки підсумок охоплює важливу інформацію

Нижче ми визначаємо інструмент `summarize_preferences`, який агент може викликати для запису компактного підсумку того, що він дізнався.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Резюме

У цьому уроці ви дізналися, як керувати контекстом у довготривалих розмовах агентів за допомогою Microsoft Agent Framework:

### Ключові концепції
- **Вікна контексту мають обмеження** — кожен токен у історії розмови коштує грошей і враховується в межі.
- **Інструменти для підсумовування** дозволяють агенту скорочувати накопичений контекст у компактні резюме, зменшуючи використання токенів, при цьому зберігаючи важливу інформацію.
- **Блокноти агента** надають постійну зовнішню пам’ять, яка зберігається незалежно від скорочення розмови.

### Що ви створили
- **Агент, що знає контекст**, який підтримує безперервність у багатокрокових розмовах
- **Інструмент підсумовування** (`summarize_preferences`), який записує ключові деталі користувача у компактному форматі
- **Багатокрокова розмова**, що демонструє збереження контексту та обробку змін

### Реальні застосування
- **Боти служби підтримки**: запам’ятовують вподобання протягом довгих сесій підтримки
- **Персональні помічники**: відстежують поточні проекти без повторного пояснення контексту
- **Освітні вчителі**: підтримують прогрес студентів протягом численних взаємодій

### Наступні кроки
- Реалізувати повноцінний інструмент блокнота з файловим збереженням
- Додати автоматичне скорочення історії після підсумовування
- Поєднати з векторними базами даних для семантичного пошуку пам’яті
- Створити агентів, які можуть продовжувати розмови через кілька днів з повним контекстом


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
